# 03 · Cuarentena y Reproceso

Este notebook maneja registros con errores críticos:
- Identifica registros que no pueden pasar a siguiente capa
- Aísla en cuarentena (tabla y archivo)
- Permite corrección manual y reintentos

**Entrada:** Errores en `raw.validacion_calidad`  
**Salida:** Registros en cuarentena en `datos/cuarentena/` y `quarantine.invoice_lines`

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_DESTINO
from datetime import datetime
import os
import pandas as pd
import json

engine_origen, engine_destino = obtener_engines()

# Rutas
DATOS_DIR = './datos'
CUARENTENA_DIR = os.path.join(DATOS_DIR, 'cuarentena')
os.makedirs(CUARENTENA_DIR, exist_ok=True)

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Timestamp: {TIMESTAMP}')
print(f'Destino: {BD_DESTINO}')
print(f'Directorio cuarentena: {CUARENTENA_DIR}')

✓ Conexión OK (origen): WideWorldImporters
✓ Conexión OK (destino): WideWorldImportersDW2
Timestamp: 20261103_112530
Destino: WideWorldImportersDW2
Directorio cuarentena: ./datos\cuarentena


## Paso 1: Crear tabla de cuarentena

In [2]:
from sqlalchemy import text

sql_crear_cuarentena = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'quarantine')
    EXEC('CREATE SCHEMA quarantine');

IF OBJECT_ID('quarantine.invoice_lines', 'U') IS NULL
BEGIN
    CREATE TABLE quarantine.invoice_lines (
        id_quarantine       INT PRIMARY KEY IDENTITY(1,1),
        id_raw              INT NOT NULL,
        lote_id             NVARCHAR(50) NOT NULL,
        payload             NVARCHAR(MAX) NOT NULL,
        errores_criticos    NVARCHAR(MAX),
        timestamp_cuarentena DATETIME DEFAULT GETDATE(),
        estado              NVARCHAR(20) DEFAULT 'PENDIENTE',
        motivo_rechazo      NVARCHAR(500),
        timestamp_resolucion DATETIME NULL,
        INDEX IX_raw (id_raw),
        INDEX IX_lote (lote_id),
        INDEX IX_estado (estado)
    );
END;
"""

try:
    with engine_destino.begin() as conn:
        conn.execute(text(sql_crear_cuarentena))
    print('[OK] Tabla quarantine.invoice_lines creada')
except Exception as e:
    print(f'[ERROR] {e}')

[OK] Tabla quarantine.invoice_lines creada


## Paso 2: Identificar registros con errores críticos

In [3]:
try:
    # Obtener registros RAW con errores críticos
    sql_errores_criticos = """
    SELECT DISTINCT
        r.id_raw,
        r.lote_id,
        r.payload,
        STRING_AGG(
            CONCAT(v.tipo_error, ': ', v.descripcion),
            ' | '
        ) as errores_concatenados
    FROM raw.invoice_lines r
    INNER JOIN raw.validacion_calidad v ON r.id_raw = v.id_raw
    WHERE v.es_critico = 1
    GROUP BY r.id_raw, r.lote_id, r.payload
    """
    
    df_criticos = pd.read_sql_query(sql_errores_criticos, engine_destino)
    print(f'[OK] Se identificaron {len(df_criticos)} registros con errores críticos')
    
    if len(df_criticos) > 0:
        print(f'\nPrimeros registros problemáticos:')
        display(df_criticos[['id_raw', 'lote_id']].head())
except Exception as e:
    print(f'[ERROR] {e}')
    df_criticos = None

[OK] Se identificaron 0 registros con errores críticos


## Paso 3: Guardar en archivo de cuarentena

In [4]:
if df_criticos is not None and len(df_criticos) > 0:
    # Archivo CSV
    archivo_cuarentena = os.path.join(CUARENTENA_DIR, f'cuarentena_invoice_{TIMESTAMP}.csv')
    df_criticos.to_csv(archivo_cuarentena, index=False, encoding='utf-8')
    print(f'[OK] Archivo de cuarentena: {archivo_cuarentena}')
    print(f'Tamaño: {os.path.getsize(archivo_cuarentena):,} bytes')
    
    # Archivo JSON con detalle
    archivo_cuarentena_json = os.path.join(CUARENTENA_DIR, f'cuarentena_invoice_{TIMESTAMP}.json')
    cuarentena_json = df_criticos.to_dict('records')
    with open(archivo_cuarentena_json, 'w', encoding='utf-8') as f:
        json.dump(cuarentena_json, f, indent=2, ensure_ascii=False)
    print(f'[OK] Archivo JSON: {archivo_cuarentena_json}')
else:
    print('[OK] No hay registros en cuarentena')

[OK] No hay registros en cuarentena


## Paso 4: Insertar en tabla de cuarentena

In [5]:
if df_criticos is not None and len(df_criticos) > 0:
    # Renombrar columnas para coincidir con BD
    df_cuarentena = df_criticos[['id_raw', 'lote_id', 'payload']].copy()
    df_cuarentena['errores_criticos'] = df_criticos['errores_concatenados']
    df_cuarentena['motivo_rechazo'] = 'Errores críticos en validación de calidad'
    df_cuarentena = df_cuarentena[['id_raw', 'lote_id', 'payload', 'errores_criticos', 'motivo_rechazo']]
    
    try:
        df_cuarentena.to_sql(
            'invoice_lines',
            engine_destino,
            schema='quarantine',
            if_exists='append',
            index=False
        )
        print(f'[OK] {len(df_cuarentena)} registros en cuarentena.invoice_lines')
    except Exception as e:
        print(f'[ERROR] {e}')

## Paso 5: Estadísticas de cuarentena

In [6]:
try:
    sql_stats = """
    SELECT 
        COUNT(*) as total_cuarentena,
        COUNT(DISTINCT lote_id) as lotes_afectados,
        COUNT(DISTINCT id_raw) as registros_unicos,
        COUNT(CASE WHEN estado = 'PENDIENTE' THEN 1 END) as pendiente_resolucion,
        COUNT(CASE WHEN estado = 'RESUELTO' THEN 1 END) as ya_resueltos
    FROM quarantine.invoice_lines
    """
    
    with engine_destino.connect() as conn:
        result = conn.execute(text(sql_stats)).mappings().first()
    
    print('[OK] Estadísticas de cuarentena:')
    if result:
        for key, val in result.items():
            print(f'  {key}: {val}')
except Exception as e:
    print(f'[ERROR] {e}')

# Listar archivos en cuarentena
print(f'\n[OK] Archivos en {CUARENTENA_DIR}:')
for archivo in os.listdir(CUARENTENA_DIR):
    ruta = os.path.join(CUARENTENA_DIR, archivo)
    tamano = os.path.getsize(ruta)
    print(f'  - {archivo} ({tamano:,} bytes)')

[OK] Estadísticas de cuarentena:
  total_cuarentena: 0
  lotes_afectados: 0
  registros_unicos: 0
  pendiente_resolucion: 0
  ya_resueltos: 0

[OK] Archivos en ./datos\cuarentena:
